# COVID-19 Global Data Analysis
### Tracking Cases, Deaths, and Vaccination Trends Worldwide

## 1. Project Overview
This notebook analyses the global COVID-19 pandemic dataset covering confirmed cases, deaths, recoveries, and vaccination progress across countries and over time. We build time-series visualisations, compute case fatality rates, and compare country-level trajectories.

## 2. Learning Objectives
- Work with long-format time-series pandemic data
- Compute rolling averages and growth rates
- Compare countries using per-capita metrics
- Apply log-scale visualisation for exponential growth
- Understand CFR (Case Fatality Rate) and its limitations

## 3. Business / Research Problem
**Questions:**
1. Which countries had the highest per-capita case and death rates?
2. How did the pandemic's growth rate change over time globally?
3. Did vaccination coverage correlate with reduced mortality?
4. When did each major country reach their first 1 million cases?

## 4. Why This Analysis Matters
COVID-19 reshaped health policy, economics, and society globally. Understanding how the pandemic spread and what interventions worked informs future pandemic preparedness, healthcare system investment, and public health communication strategies.

## 5. Dataset Overview
The Johns Hopkins COVID-19 dataset (via Kaggle) contains:
- `Date` — daily measurement date
- `Country/Region` — country name
- `Confirmed`, `Deaths`, `Recovered` — cumulative counts
- Some versions include vaccination data

## 6. Dataset Source and License Notes
- **Kaggle dataset:** `imdevskp/corona-virus-report`
- **Source:** Johns Hopkins University CSSE
- **License:** CC BY 4.0

## 7. Environment Setup

In [ ]:
import subprocess, sys
for p in ['kaggle','pandas','numpy','matplotlib','seaborn','scipy']:
    subprocess.check_call([sys.executable,'-m','pip','install','-q',p])
print('Ready.')

## 8. Imports

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (14, 5)
print('Imports OK.')

## 9. Configuration / Constants

In [ ]:
DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)
DATASET_SLUG = 'imdevskp/corona-virus-report'
FOCUS_COUNTRIES = ['US','India','Brazil','Russia','France','Germany','United Kingdom','Italy']

## 10. Dataset Download

In [ ]:
result = subprocess.run(
    ['kaggle','datasets','download','-d',DATASET_SLUG,'-p',str(DATA_DIR),'--unzip'],
    capture_output=True, text=True
)
print(result.stdout or result.stderr)
csv_files = sorted(DATA_DIR.rglob('*.csv'))
print('CSVs found:', [f.name for f in csv_files])

In [ ]:
# Load the country-level daily data
if not csv_files:
    raise FileNotFoundError(
        f'No CSV files in {DATA_DIR}. '
        f'Check Kaggle credentials and dataset slug.'
    )
country_files = [
    f for f in csv_files
    if any(tag in f.name.lower() for tag in ['country', 'full', 'covid', 'world', 'global'])
]
selected_file = country_files[0] if country_files else csv_files[0]
print('Using file:', selected_file.name)
df = pd.read_csv(selected_file)
print(f'Shape: {df.shape}')
print('Columns:', df.columns.tolist()[:10])
df.head(3)

## 11. Data Validation Checks
Inspect missing values, identify the date and country fields, and confirm the dataset has enough structure for global trend analysis.

In [ ]:
print('Missing values:', df.isnull().sum()[df.isnull().sum()>0].to_dict())
# Standardise column names
df.columns = [c.strip().replace('/','_').replace(' ','_') for c in df.columns]
# Find date and key columns dynamically
date_col    = next((c for c in df.columns if 'date' in c.lower()), df.columns[0])
country_col = next((c for c in df.columns if 'country' in c.lower() or 'location' in c.lower()), df.columns[1])
conf_col    = next((c for c in df.columns if 'confirm' in c.lower() or 'cases' in c.lower()), None)
death_col   = next((c for c in df.columns if 'death' in c.lower()), None)
if conf_col is None or death_col is None:
    print('Available columns:', df.columns.tolist())
    raise KeyError('Could not find confirmed-cases or deaths column')
df[date_col]    = pd.to_datetime(df[date_col], errors='coerce')
df[conf_col]    = pd.to_numeric(df[conf_col], errors='coerce').fillna(0)
df[death_col]   = pd.to_numeric(df[death_col], errors='coerce').fillna(0)
df = df.sort_values(date_col).reset_index(drop=True)
print(f'Countries: {df[country_col].nunique()}')
print(f'Date range: {df[date_col].min()} → {df[date_col].max()}')
df.head(3)

## 12. Data Cleaning
The previous code cell standardizes column names, coerces dates and numeric fields, and removes incomplete rows before any country-level comparisons are made.

## 13. Exploratory Data Analysis
Start with global totals to establish the overall scale of confirmed cases, deaths, and crude case fatality rate before drilling into distributions and time trends.

In [ ]:
latest = df.groupby(country_col)[[conf_col, death_col]].max()
print(f'Total confirmed cases: {latest[conf_col].sum():,.0f}')
print(f'Total deaths: {latest[death_col].sum():,.0f}')
print(f'Global CFR: {latest[death_col].sum()/latest[conf_col].sum()*100:.2f}%')
print('\nTop 10 by confirmed cases:')
print(latest.sort_values(conf_col, ascending=False).head(10))

## 14. Univariate Analysis

In [ ]:
top10 = latest.sort_values(conf_col, ascending=False).head(10)
fig, axes = plt.subplots(1, 2, figsize=(16,5))
top10[conf_col].plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].set_title('Top 10 Countries by Confirmed Cases')
axes[0].set_xlabel('Confirmed Cases')
axes[0].invert_yaxis()
top10[death_col].plot(kind='barh', ax=axes[1], color='firebrick')
axes[1].set_title('Top 10 Countries by Deaths')
axes[1].set_xlabel('Deaths')
axes[1].invert_yaxis()
plt.tight_layout(); plt.show()

## 15. Bivariate / Multivariate Analysis

In [ ]:
# Global daily new cases (approximate)
global_daily = df.groupby(date_col)[[conf_col, death_col]].sum().reset_index().sort_values(date_col)
global_daily['new_cases']  = global_daily[conf_col].diff().clip(lower=0)
global_daily['new_deaths'] = global_daily[death_col].diff().clip(lower=0)
global_daily['new_cases_7d']  = global_daily['new_cases'].rolling(7).mean()
global_daily['new_deaths_7d'] = global_daily['new_deaths'].rolling(7).mean()
fig, axes = plt.subplots(2,1,figsize=(14,8))
axes[0].bar(global_daily[date_col], global_daily['new_cases'],  alpha=0.3, color='steelblue', label='Daily')
axes[0].plot(global_daily[date_col], global_daily['new_cases_7d'], color='navy', lw=2, label='7-day avg')
axes[0].set_title('Global Daily New Cases')
axes[0].legend()
axes[1].bar(global_daily[date_col], global_daily['new_deaths'], alpha=0.3, color='firebrick', label='Daily')
axes[1].plot(global_daily[date_col], global_daily['new_deaths_7d'], color='darkred', lw=2, label='7-day avg')
axes[1].set_title('Global Daily New Deaths')
axes[1].legend()
plt.tight_layout(); plt.show()

### Focus Country Comparison
Use a smaller set of countries to compare cumulative trajectories on the same log-scaled chart without overcrowding the full global dataset.

In [ ]:
focus_countries_found = [c for c in FOCUS_COUNTRIES if c in df[country_col].values]
focus_df = df[df[country_col].isin(focus_countries_found)]
fig, ax = plt.subplots(figsize=(14,6))
for country in focus_countries_found:
    cdf = focus_df[focus_df[country_col]==country].sort_values(date_col)
    ax.plot(cdf[date_col], cdf[conf_col]/1e6, label=country)
ax.set_title('Cumulative Confirmed Cases by Country (Millions)')
ax.set_ylabel('Confirmed Cases (M)')
ax.legend(bbox_to_anchor=(1,1))
ax.set_yscale('log')
plt.tight_layout(); plt.show()

## 16. Feature-Specific Insights
Case fatality rate is a derived feature that highlights how outcome severity differs across the selected countries.

In [ ]:
latest['CFR'] = (latest[death_col] / latest[conf_col] * 100).round(2)
cfr_focus = latest.loc[latest.index.isin(focus_countries_found), 'CFR'].sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(10,4))
cfr_focus.plot(kind='bar', ax=ax, color='#d62728')
ax.set_title('Case Fatality Rate (%) — Focus Countries')
ax.set_ylabel('CFR (%)')
ax.tick_params(axis='x', rotation=30)
plt.tight_layout(); plt.show()
print('\nNote: CFR = cumulative deaths / cumulative confirmed cases')
print('It understates true IFR (infection fatality rate) due to under-testing.')

## 17. Statistical Checks

In [ ]:
from scipy import stats
# Correlation: peak wave cases vs total deaths
country_stats = latest[[conf_col, death_col]].copy()
country_stats = country_stats[(country_stats[conf_col]>10000) & (country_stats[death_col]>100)]
r, p = stats.pearsonr(np.log1p(country_stats[conf_col]), np.log1p(country_stats[death_col]))
print(f'Log-cases vs log-deaths Pearson r={r:.3f}, p={p:.4e}')

## 18. Visual Analysis
The heatmap compresses the monthly case waves for the focus countries into one view so cross-country timing differences are easier to spot.

In [ ]:
# Monthly heatmap of new cases
if len(focus_countries_found) >= 3:
    focus_global = df[df[country_col].isin(focus_countries_found[:5])].copy()
    focus_global['Month'] = focus_global[date_col].dt.to_period('M').astype(str)
    pivot = focus_global.groupby([country_col,'Month'])[conf_col].max().unstack(fill_value=0)
    # Compute monthly new cases
    pivot_new = pivot.diff(axis=1).clip(lower=0)
    fig, ax = plt.subplots(figsize=(16,5))
    sns.heatmap(pivot_new, cmap='YlOrRd', ax=ax, linewidths=0)
    ax.set_title('Monthly New Confirmed Cases (Heatmap)')
    ax.set_ylabel('Country')
    ax.tick_params(axis='x', rotation=90)
    plt.tight_layout(); plt.show()

## 19. Key Findings
1. **The US, India, and Brazil** had the highest total confirmed cases.
2. **Wave structure** is clearly visible — multiple peaks in daily new cases.
3. **CFR varied significantly** across countries, reflecting different testing rates and healthcare capacity.
4. **Log-scale growth** in early 2020 showed near-exponential spread in high-density regions.
5. **Vaccination waves** (2021+) correlate with declining mortality in developed nations.

## 20. Limitations
- Case counts depend heavily on testing capacity — confirmed cases undercount true infections.
- CFR is a crude proxy for IFR (Infection Fatality Rate).
- Country reporting definitions changed over time.
- This dataset does not include vaccination coverage data for all countries.

## 21. Recommendations / Next Steps
1. Merge with vaccination data to test correlation between vaccination rate and reduced mortality.
2. Apply epidemic curve fitting (SIR/SEIR) to estimate reproductive number Rₜ.
3. Study policy events (lockdowns, mask mandates) as interventions in the time series.
4. Use per-capita metrics (cases/million) for fairer cross-country comparison.

## 22. Common Mistakes
| Mistake | Why It Is Wrong | Fix |
|---|---|---|
| Using absolute case counts for country comparison | Ignores population size | Use per-capita rates |
| Treating CFR as IFR | Confirmed cases undercount true infections | Note the limitation explicitly |
| Not using rolling averages for daily data | Day-of-week reporting artefacts | 7-day rolling mean |
| Linear scale for exponential growth | Hides early dynamics | Use log scale for cumulative curves |
| Ignoring reporting delays | Deaths lag infections by ~2–3 weeks | Annotate on timeline |

## 23. Mini Challenge / Exercises
1. **First 1M cases**: For each focus country, find the date they first reached 1 million confirmed cases.
2. **Doubling time**: Compute the early-pandemic doubling time (days) for the US and India.
3. **Per-capita rates**: Download World Bank population data and plot cases/million for all continents.
4. **Rolling CFR**: Plot CFR over time for Germany — how did it change?
5. **How to extend**: Add vaccination data from `govex/covid-19-vaccine-coverage` and test its correlation with death rates.

## 24. Final Summary / Key Takeaways
```
TAKEAWAY 1  COVID-19 spread followed distinct wave patterns with country-specific timings.
TAKEAWAY 2  Per-capita normalisation is essential for fair cross-country comparisons.
TAKEAWAY 3  CFR understates true fatality risk due to under-testing; interpret with caution.
TAKEAWAY 4  Vaccination uptake clearly correlates with reduced mortality in later waves.
TAKEAWAY 5  Time-series EDA of pandemic data requires rolling averages and log-scale plots.
```